# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

In [12]:
import os
import time
import requests
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

### Paso 1: Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`

In [13]:
def paso1_tfidf_turismo(ruta_archivo):
    print("--- PASO 1: TF-IDF Corpus Turismo ---")
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        lineas = f.readlines()
    
    # Limpieza básica por si copiaste las etiquetas "" del chat
    corpus_turismo = [re.sub(r'\\s*', '', linea).strip() for linea in lineas if linea.strip()]
    
    vectorizer_turismo = TfidfVectorizer()
    matriz_tfidf_turismo = vectorizer_turismo.fit_transform(corpus_turismo)
    
    print(f"Corpus Turismo procesado: {len(corpus_turismo)} documentos.")
    print(f"Dimensiones de la matriz TF-IDF: {matriz_tfidf_turismo.shape}")
    print(f"Vocabulario extraído: {len(vectorizer_turismo.get_feature_names_out())} términos.\n")
    
    return vectorizer_turismo, matriz_tfidf_turismo

### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

In [14]:
def paso2_construir_gutenberg(num_libros=1000, carpeta_destino="data/gutenberg_1000"):
    print("--- PASO 2: Construyendo Corpus Gutenberg ---")
    if not os.path.exists(carpeta_destino):
        os.makedirs(carpeta_destino)
    
    libros_descargados = len(os.listdir(carpeta_destino))
    if libros_descargados >= num_libros:
        print(f"Ya existen {libros_descargados} libros en {carpeta_destino}. Se omite la descarga.\n")
        return
    
    print(f"Descargando {num_libros} libros (esto tomará algo de tiempo)...")
    
    # Iteramos sobre IDs de Project Gutenberg
    libros_exitosos = 0
    id_libro = 1
    
    while libros_exitosos < num_libros:
        ruta_guardado = os.path.join(carpeta_destino, f"libro_{id_libro}.txt")
        
        # Si ya existe localmente, lo saltamos
        if os.path.exists(ruta_guardado):
            libros_exitosos += 1
            id_libro += 1
            continue
            
        url = f"https://www.gutenberg.org/cache/epub/{id_libro}/pg{id_libro}.txt"
        try:
            respuesta = requests.get(url, timeout=5)
            # Solo guardamos si la petición fue exitosa y es texto plano
            if respuesta.status_code == 200 and "text/plain" in respuesta.headers.get("Content-Type", ""):
                with open(ruta_guardado, 'w', encoding='utf-8') as f:
                    f.write(respuesta.text)
                libros_exitosos += 1
                if libros_exitosos % 50 == 0:
                    print(f"Descargados {libros_exitosos}/{num_libros} libros...")
            time.sleep(0.5) # Pausa para no saturar el servidor
        except requests.RequestException:
            pass # Ignoramos errores de conexión y probamos con el siguiente ID
            
        id_libro += 1
    print("Descarga del corpus Gutenberg completada.\n")

### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

In [15]:
def paso3_tfidf_gutenberg(carpeta_origen="data/gutenberg_1000"):
    print("--- PASO 3: TF-IDF Corpus Gutenberg ---")
    archivos = [os.path.join(carpeta_origen, f) for f in os.listdir(carpeta_origen) if f.endswith('.txt')]
    
    corpus_gutenberg = []
    nombres_documentos = []
    
    print("Cargando libros en memoria...")
    for archivo in archivos:
        with open(archivo, 'r', encoding='utf-8', errors='ignore') as f:
            # Tomamos los primeros 5000 caracteres para evitar colapsar la RAM durante el ejercicio
            texto = f.read(5000) 
            corpus_gutenberg.append(texto)
            nombres_documentos.append(os.path.basename(archivo))
            
    # Usamos stop_words en inglés ya que la mayoría de Gutenberg está en ese idioma
    vectorizer_gutenberg = TfidfVectorizer(stop_words='english', max_features=10000)
    matriz_tfidf_gutenberg = vectorizer_gutenberg.fit_transform(corpus_gutenberg)
    
    print(f"Corpus Gutenberg procesado: {len(corpus_gutenberg)} documentos.")
    print(f"Dimensiones de la matriz TF-IDF: {matriz_tfidf_gutenberg.shape}\n")
    
    return vectorizer_gutenberg, matriz_tfidf_gutenberg, nombres_documentos

### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

In [16]:
def buscar(consulta, vectorizer, matriz_tfidf, nombres_documentos, top_k=5):
    """
    Transforma la consulta en un vector y calcula la similitud del coseno
    contra todos los documentos de la matriz TF-IDF.
    """
    # 1. Transformar la consulta al mismo espacio vectorial
    # OJO: se usa transform(), no fit_transform()
    vector_consulta = vectorizer.transform([consulta])
    
    # 2. Calcular similitud del coseno (devuelve una matriz de 1 x N)
    similitudes = cosine_similarity(vector_consulta, matriz_tfidf).flatten()
    
    # 3. Obtener los índices de los 'top_k' documentos más similares
    indices_top = similitudes.argsort()[-top_k:][::-1]
    
    print(f"\n--- Resultados de búsqueda para: '{consulta}' ---")
    for i in indices_top:
        similitud = similitudes[i]
        if similitud > 0:
            print(f"Documento: {nombres_documentos[i]} | Similitud: {similitud:.4f}")
        else:
            print("No se encontraron más coincidencias relevantes (Similitud = 0).")
            break

### Ejecución principal

In [ ]:
if __name__ == "__main__":
    import os
    
    # Ruta inteligente
    directorio_base = os.getcwd()
    if os.path.basename(directorio_base) != 'work' and os.path.exists('work'):
        os.chdir('work')
        directorio_base = os.getcwd()
        
    # Importación sin importar el equipo donde se ejecute el código
    ruta_turismo = os.path.join(directorio_base, "data", "01_corpus_turismo_500.txt")
    
    # Verifica si el archivo existe antes de ejecutar
    if not os.path.exists(ruta_turismo):
        print(f"Error: No se encontró el archivo '{ruta_turismo}'.")
    else:
        print("¡Archivo encontrado! Iniciando proceso...")
        # Paso 1
        vec_tur, mat_tur = paso1_tfidf_turismo(ruta_turismo)
        
        # Paso 2
        paso2_construir_gutenberg(num_libros=1000) 
        
        # Paso 3
        vec_gut, mat_gut, nombres_gut = paso3_tfidf_gutenberg()
        
        # Paso 4
        consulta_prueba = "science fiction adventure"
        buscar(consulta_prueba, vec_gut, mat_gut, nombres_gut)

¡Archivo encontrado! Iniciando proceso...
--- PASO 1: TF-IDF Corpus Turismo ---
Corpus Turismo procesado: 500 documentos.
Dimensiones de la matriz TF-IDF: (500, 116)
Vocabulario extraído: 116 términos.

--- PASO 2: Construyendo Corpus Gutenberg ---
Descargando 1000 libros (esto tomará algo de tiempo)...
Descargados 50/1000 libros...
Descargados 100/1000 libros...
Descargados 150/1000 libros...
Descargados 200/1000 libros...
Descargados 250/1000 libros...
Descargados 300/1000 libros...
Descargados 350/1000 libros...
Descargados 400/1000 libros...
Descargados 450/1000 libros...
Descargados 500/1000 libros...
Descargados 550/1000 libros...
Descargados 600/1000 libros...
Descargados 650/1000 libros...
Descargados 700/1000 libros...
Descargados 750/1000 libros...
Descargados 800/1000 libros...
Descargados 850/1000 libros...
Descargados 900/1000 libros...
Descargados 950/1000 libros...
Descargados 1000/1000 libros...
Descarga del corpus Gutenberg completada.

--- PASO 3: TF-IDF Corpus Gutenb